# SCRES-IA — Variante FINA (por-lote, 24 decisiones): ¿un espacio de decisión más rico deja ganar a RL?

Este es el **segundo** notebook, la **puerta C6** del plan. David tenía una intuición: que el espacio
de decisión de 8 decisiones semanales es **demasiado grueso**, y que decisiones **más finas** le darían
a RL margen para ganarle al MPC. Aquí lo probamos.

**Qué cambia vs el notebook principal:** en vez de 8 decisiones semanales de "cuántos de 3 lotes"
(4⁸ = 65 536), aquí hay **24 decisiones binarias, una por lote físico** (P_C/P_H). Eso da
**2²⁴ = 16 777 216** configuraciones — mucho más expresivo, y **24 pasos** (el frame-stack puede ser
24 con frames reales, lo que David quería).

> **⚠️ Lee esto: qué NO es este notebook.**
> - **Ya NO hay clave de respuestas exacta.** 2²⁴ no es enumerable; `H_learned` se mide contra una
>   **muestra grande** de la frontera (una cota, no el óptimo exacto). Es honesto, pero no es la
>   frontera exacta del paper.
> - **NO es directamente comparable con RecurrentPPO** — es otro entorno (más rico).
> - **El clásico belief-MPC aquí sigue en el espacio grueso** (conteo semanal, 4 acciones): NO puede
>   hacer patrones por-lote arbitrarios. Así que si RL le gana, es evidencia de que **"acciones más
>   ricas ayudan"** — todavía NO de que "la red le gana a la estructura con las mismas armas" (para eso
>   haría falta un MPC por-lote, trabajo futuro). Aun así, ganarle al clásico con un espacio más rico
>   sería la **primera señal** de la hipótesis de David.
>
> Úsalo para **descubrir** si hay señal en las decisiones finas. Si la hay, lo formalizamos con un MPC
> justo y frontera muestreada rigurosa.


## Instrucciones
- **Colab:** https://colab.research.google.com/github/Thom-320/scres-ia/blob/qr1-c1-natural-continuation/notebooks/scresia_david_perbatch_C6_FINAL.ipynb
- **Kaggle:** activa **Internet: On** + GPU, o importa la URL de GitHub.
- **`Run all`.** Defaults listos (`serious`, 200k timesteps, celda rho90_share90, tu transformer, PPO).
- **Tiempo estimado (default PPO):** ~**25 min en CPU** (medido: ~155 steps/s → 200k ≈ 21 min +
  baselines ≈ 4 min); menos en GPU. SAC/off-policy más lento.
- **✅ PUEDES CAMBIAR** (celda de config + celda de arquitectura): `ALGORITHM` (PPO/RecurrentPPO/SAC/
  QRDQN/A2C/DQN), tu arquitectura, `FRAME_STACK` (aquí SÍ hasta 24 = episodio completo), `CELL_ID`,
  hiperparámetros. **⛔ NO CAMBIES** el entorno por-lote, cómo se calculan la frontera muestreada y el
  clásico, ni la lógica del veredicto.


## 1) Configuración  ·  ✅ **PUEDES CAMBIAR**

In [ ]:

RUN_PROFILE = "serious"          # "serious" (real, DEFAULT) | "debug" (smoke ~2min)
ARCH_VARIANT = "positional_v1_1" # "original_v1_0" | "positional_v1_1" | "mlp_baseline"
ALGORITHM    = "PPO"             # "PPO" | "RecurrentPPO" | "SAC" | "QRDQN" | "A2C" | "DQN"
FRAME_STACK  = 24                # aqui el episodio dura 24 pasos -> 24 = campaña completa (frames reales)
FEATURES_DIM = 120
CELL_ID = "rho90_share90"        # "rho75_share90" | "rho90_share75" | "rho90_share90"
TRAIN_TAPE_START = 971_000_000   # namespace de DESARROLLO (distinto del notebook 1)
EVAL_TAPE_START  = 981_000_000
SEED = 42
SAMPLED_FRONTIER = 100_000       # muestras aleatorias de las 2^24 config. (cota de H_learned, NO exacta)

TOTAL_TIMESTEPS_DEBUG, EVAL_TAPES_DEBUG     = 5_000,   8
TOTAL_TIMESTEPS_SERIOUS, EVAL_TAPES_SERIOUS = 200_000, 64
GIT_URL, GIT_BRANCH = "https://github.com/Thom-320/scres-ia.git", "qr1-c1-natural-continuation"
if RUN_PROFILE=="debug":   TOTAL_TIMESTEPS,EVAL_TAPES,SAMPLED_FRONTIER = TOTAL_TIMESTEPS_DEBUG,EVAL_TAPES_DEBUG,5_000
elif RUN_PROFILE=="serious":TOTAL_TIMESTEPS,EVAL_TAPES = TOTAL_TIMESTEPS_SERIOUS,EVAL_TAPES_SERIOUS
else: raise ValueError("RUN_PROFILE debe ser serious/debug")
TRAIN_TAPE_END = TRAIN_TAPE_START + max(TOTAL_TIMESTEPS//24 + 50_000, 50_000)
EVAL_TAPE_END  = EVAL_TAPE_START + EVAL_TAPES - 1
print({"profile":RUN_PROFILE,"algo":ALGORITHM,"arch":ARCH_VARIANT,"cell":CELL_ID,
       "frame_stack":FRAME_STACK,"timesteps":TOTAL_TIMESTEPS,"eval_tapes":EVAL_TAPES,"sampled_frontier":SAMPLED_FRONTIER})

## 2) Repositorio + dependencias

In [ ]:

import os, sys, shutil, subprocess, json, math, time
from pathlib import Path
import numpy as np, pandas as pd
IN_COLAB="google.colab" in sys.modules or Path("/content").exists(); IN_KAGGLE=Path("/kaggle/working").exists()
PKGS=["simpy>=4.1","numpy>=1.26","gymnasium>=0.29","stable-baselines3>=2.3","sb3-contrib>=2.3","torch>=2.1","einops>=0.7","pandas>=2.2"]
def run(cmd):
    print("$"," ".join(map(str,cmd))); r=subprocess.run(cmd,text=True,capture_output=True)
    print((r.stdout or "")[-1200:]); print((r.stderr or "")[-1200:])
    if r.returncode: raise RuntimeError(cmd)
REPO=Path("/kaggle/working/scresia") if IN_KAGGLE else Path("/content/scresia") if IN_COLAB else Path.cwd()
if IN_COLAB or IN_KAGGLE:
    run([sys.executable,"-m","pip","install","-q",*PKGS]); shutil.rmtree(REPO,ignore_errors=True)
    run(["git","clone","--depth","1","--branch",GIT_BRANCH,GIT_URL,str(REPO)])
sys.path.insert(0,str(REPO)); print("repo:",REPO,"| env code:",(REPO/"supply_chain").exists())

## 3) Benchmark de referencia (constantes)  ·  ⛔ **NO CAMBIAR**

In [ ]:

# En el entorno GRUESO (8 decisiones) RecurrentPPO EMPATO al clasico (H_neural ~= 0). La pregunta de
# ESTE notebook: con acciones por-lote (mas ricas), ¿RL le GANA al MISMO clasico belief-MPC?
FROZEN_RECURRENTPPO = {"rho75_share90":-0.00159,"rho90_share75":-0.00072,"rho90_share90":-0.00041}  # H_neural (grueso)
WIN_BAR_H_NEURAL_LCB=0.01; WORST_PRODUCT_MARGIN=-0.02
print("En el entorno grueso, RecurrentPPO empato al clasico en", CELL_ID, "->", FROZEN_RECURRENTPPO[CELL_ID])
print("Objetivo aqui: H_neural (por-lote) con LCB95 >= +0.01 sobre el clasico (con el caveat de accion-mas-rica).")

## 4) Entorno POR-LOTE (24 decisiones)  ·  ⛔ **NO CAMBIAR**

In [ ]:
# ⛔ NO CAMBIAR: entorno de la variante fina C6 (24 decisiones binarias -> 8 patrones -> DES).
"""PerBatchRetEnv: finer-transition C6 variant. 24 per-batch binary decisions."""
import json, numpy as np
import gymnasium as gym
from gymnasium import spaces
from supply_chain.program_o_full_des_transducer import (
    FullDESSkeleton, extract_full_des_skeleton, simulate_full_des_frontier)
from supply_chain.program_o_state_rich import (
    StateRichConfiguration, StateRichObservation, state_rich_calendar)
from supply_chain.program_o_ret_env import normalized_state_rich_observation, ProgramORetCell

# scheduler de 8 patrones: accion 0..7 = patron binario de los 3 lotes semanales (para el DES)
SCHED_PATTERN = {str(i): [("P_C" if (i>>j)&1 else "P_H") for j in range(3)] for i in range(8)}
# scheduler de conteo (4 acciones) SOLO para la observacion state-rich (que solo depende del
# numero de P_C en la semana, no del orden intra-semana). Se carga del contrato en __init__.
_CFG = StateRichConfiguration("belief_mpc", 3)
PERBATCH_OBS_DIM = 21 + 3 + 3   # obs semanal state-rich + posicion-en-semana (one-hot 3) + bits ya elegidos (3)

class PerBatchRetEnv(gym.Env):
    """24 decisiones binarias (una por lote fisico). Cada 3 forman un patron semanal
    (0..7) que alimenta el DES. Observacion = obs semanal state-rich + posicion + bits
    intra-semana ya elegidos."""
    def __init__(self, tape_seed_start, tape_seed_end,
                 cell=ProgramORetCell("rho90_share90",0.90,0.90),
                 belief_persistence=0.75, belief_share=0.90, count_scheduler=None):
        super().__init__()
        self.tape_seed_start=int(tape_seed_start); self.tape_seed_end=int(tape_seed_end)
        self.cell=cell; self.bp=float(belief_persistence); self.bs=float(belief_share)
        # scheduler de conteo (4 acciones) para la observacion; si no se pasa, se construye uno
        # canonico (accion k = k lotes P_C, los primeros k) — mismo mapeo conteo->obs.
        self.sched_count = count_scheduler or {str(k): ["P_C"]*k + ["P_H"]*(3-k) for k in range(4)}
        self.action_space=spaces.Discrete(2)  # 0=P_H, 1=P_C para el proximo lote
        self.observation_space=spaces.Box(0.0,1.0,(PERBATCH_OBS_DIM,),np.float32)
        self._ep=0; self._batch_bits=[]; self._skeleton=None
        self._cached_week=-1; self._cached_base=None

    def _skel(self, seed):
        sk,_=extract_full_des_skeleton(seed=int(seed),scheduler=SCHED_PATTERN,
            regime_persistence=self.cell.regime_persistence,dominant_share=self.cell.dominant_share,
            downstream_freight_physics_mode="fixed_clock_physical_v1")
        return sk

    def _weekly_actions(self):
        # bits ya completos -> acciones semanales 0..7; semanas incompletas -> 0 (placeholder)
        wa=[]
        for w in range(8):
            b=self._batch_bits[3*w:3*w+3]
            if len(b)==3: wa.append(int(b[0]|(b[1]<<1)|(b[2]<<2)))
            else: wa.append(0)
        return wa

    def _weekly_counts(self):
        # numero de P_C por semana (0..3); semanas incompletas -> 0 (placeholder)
        wc=[]
        for w in range(8):
            b=self._batch_bits[3*w:3*w+3]
            wc.append(int(sum(b)) if len(b)==3 else 0)
        return wc

    def _obs(self):
        n=len(self._batch_bits); week=n//3; pos=n%3
        # la obs semanal solo depende de las semanas YA completas -> cachear por semana
        if self._cached_week != week:
            counts=self._weekly_counts()
            _cal, decisions = state_rich_calendar(skeleton=self._skeleton.as_dict(),scheduler=self.sched_count,
                config=_CFG, regime_persistence=self.bp, dominant_share=self.bs, action_overrides=tuple(counts))
            self._cached_base=normalized_state_rich_observation(decisions[min(week,7)].observation).astype(np.float32)
            self._cached_week=week
        base=self._cached_base   # 21-d obs semanal (cacheada)
        posv=np.zeros(3,np.float32); posv[pos]=1.0
        bits=np.zeros(3,np.float32)
        for j in range(pos): bits[j]=float(self._batch_bits[3*week+j])
        return np.concatenate([base.astype(np.float32),posv,bits]).astype(np.float32)

    def _tape_seed(self):
        c=self.tape_seed_end-self.tape_seed_start+1
        if self._ep>=c: raise RuntimeError("per-batch tape namespace exhausted")
        return self.tape_seed_start+self._ep

    def reset(self,*,seed=None,options=None):
        super().reset(seed=seed); options=options or {}
        ts=int(options["tape_seed"]) if "tape_seed" in options else self._tape_seed()
        self._skeleton=self._skel(ts); self._batch_bits=[]; self._ep+=1
        self._cached_week=-1; self._cached_base=None   # invalidar cache de obs
        return self._obs(), {"tape_seed":ts}

    def step(self, action):
        self._batch_bits.append(int(action))
        if len(self._batch_bits)<24:
            return self._obs(), 0.0, False, False, {}
        wa=self._weekly_actions()
        m=simulate_full_des_frontier(skeleton=self._skeleton,scheduler=SCHED_PATTERN,
            calendars=np.asarray([wa],np.uint8))
        met={k:float(v[0]) for k,v in m.items()}
        obs=np.zeros(self.observation_space.shape,np.float32)
        return obs, float(met["ret_visible"]), True, False, {
            "weekly_actions":wa,"batch_bits":list(self._batch_bits),"metrics":met}

parent = json.loads((REPO/'contracts/program_o_full_des_hpi_translation_v1.json').read_text())
SCHED_COUNT = parent['action']['within_week_schedulers'][parent['action']['primary_scheduler']]
CELLS = {c.cell_id:c for c in (ProgramORetCell('rho75_share90',0.75,0.90),
         ProgramORetCell('rho90_share75',0.90,0.75), ProgramORetCell('rho90_share90',0.90,0.90))}
CELL = CELLS[CELL_ID]
print('entorno por-lote listo. obs_dim =', PERBATCH_OBS_DIM, '| accion = Discrete(2) x 24 pasos')

## 5) Tu arquitectura  ·  ✅ **EDITA/REVISA AQUÍ** (misma que el notebook 1; funciona en obs 27-D)

In [ ]:

import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack
from stable_baselines3.common.monitor import Monitor
import torch
from einops import rearrange
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor

class DMLPAPositionalV11(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor, features_dim=120):
        super().__init__(observation_space, features_dim)
        self.factor=factor; self.obs_dimension=observation_space.shape[0]//factor
        self.latent_rw=torch.nn.Sequential(torch.nn.Linear(self.obs_dimension,100),torch.nn.GELU(),torch.nn.Linear(100,features_dim))
        self.pre_norm=torch.nn.LayerNorm(features_dim)
        L=torch.nn.TransformerEncoderLayer(d_model=features_dim,nhead=12,batch_first=True)
        self.accumulated=torch.nn.TransformerEncoder(L,num_layers=4)
        self.register_buffer("pe",self._pe(factor,features_dim))
    @staticmethod
    def _pe(n,d):
        pe=torch.zeros(n,d); pos=torch.arange(0,n).unsqueeze(1); div=torch.exp(torch.arange(0,d,2)*(-math.log(10000.0)/d))
        pe[:,0::2]=torch.sin(pos*div); pe[:,1::2]=torch.cos(pos*div); return pe.unsqueeze(0)
    def forward(self,x):
        x=rearrange(x,"b (d k) -> b d k",d=self.factor); x=self.latent_rw(x)+self.pe; x=self.pre_norm(x); x=self.accumulated(x); return x[:,-1,:]

class DMLPAOriginal(BaseFeaturesExtractor):
    def __init__(self, observation_space, factor, features_dim=120):
        super().__init__(observation_space, features_dim)
        self.factor=factor; self.obs_dimension=observation_space.shape[0]//factor
        self.latent_rw=torch.nn.Sequential(torch.nn.Linear(self.obs_dimension,100),torch.nn.GELU(),torch.nn.Linear(100,features_dim))
        L=torch.nn.TransformerEncoderLayer(d_model=features_dim,nhead=12,batch_first=True)
        self.accumulated=torch.nn.TransformerEncoder(L,num_layers=4)
    def forward(self,x):
        x=rearrange(x,"b (d k) -> b d k",d=self.factor); x=self.latent_rw(x); x=self.accumulated(x); return x[:,-1,:]

ARCHS={"original_v1_0":DMLPAOriginal,"positional_v1_1":DMLPAPositionalV11,"mlp_baseline":None}
ARCH=ARCHS[ARCH_VARIANT]

# SAC continuo: Box(2)->argmax
class BoxToDiscrete(gym.ActionWrapper):
    def __init__(self,env,n=2): super().__init__(env); self.action_space=spaces.Box(-1.0,1.0,(n,),np.float32)
    def action(self,a): return int(np.argmax(np.asarray(a).ravel()))
CONTINUOUS={"SAC"}
def make_vec(a,b):
    def _mk():
        e=PerBatchRetEnv(a,b,cell=CELL)
        return Monitor(BoxToDiscrete(e,2) if ALGORITHM in CONTINUOUS else e)
    return VecFrameStack(DummyVecEnv([_mk]),n_stack=FRAME_STACK)
train_env=make_vec(TRAIN_TAPE_START,TRAIN_TAPE_END)
if ARCH is not None:
    _ex=ARCH(train_env.observation_space,factor=FRAME_STACK,features_dim=FEATURES_DIM)
    with torch.no_grad(): print("arquitectura OK, salida:",_ex(torch.as_tensor(train_env.reset()).float()).shape)

## 6) Entrenador multi-algoritmo  ·  ✅ hiperparámetros ajustables

In [ ]:

from stable_baselines3 import PPO, A2C, DQN, SAC
from sb3_contrib import RecurrentPPO, QRDQN
def pk():
    return {} if ARCH is None else dict(features_extractor_class=ARCH,features_extractor_kwargs=dict(features_dim=FEATURES_DIM,factor=FRAME_STACK))
def build(env):
    c=dict(policy_kwargs=pk(),device="auto",verbose=1,seed=SEED)
    if ALGORITHM=="PPO":          return PPO("MlpPolicy",env,n_steps=768,batch_size=96,n_epochs=10,gamma=0.99,**c)
    if ALGORITHM=="A2C":          return A2C("MlpPolicy",env,n_steps=24,gamma=0.99,**c)
    if ALGORITHM=="RecurrentPPO": return RecurrentPPO("MlpLstmPolicy",env,n_steps=768,batch_size=96,n_epochs=10,gamma=0.99,**c)
    if ALGORITHM=="DQN":          return DQN("MlpPolicy",env,learning_starts=1000,buffer_size=50_000,train_freq=24,gamma=0.99,**c)
    if ALGORITHM=="QRDQN":        return QRDQN("MlpPolicy",env,learning_starts=1000,buffer_size=50_000,train_freq=24,gamma=0.99,**c)
    if ALGORITHM=="SAC":          return SAC("MlpPolicy",env,learning_starts=1000,buffer_size=50_000,train_freq=1,gamma=0.99,**c)
    raise ValueError(ALGORITHM)
model=build(train_env); IS_RECURRENT=ALGORITHM=="RecurrentPPO"
t0=time.time(); model.learn(total_timesteps=TOTAL_TIMESTEPS); print("entrenamiento (s):",round(time.time()-t0,1))

## 7) Líneas base: frontera MUESTREADA (cota) + clásico  ·  ⛔ **NO CAMBIAR**

In [ ]:

# ⛔ NO CAMBIAR. Frontera: 2^24 no es enumerable -> muestra grande (COTA de H_learned, no exacta).
# Clasico: el MISMO belief-MPC del notebook 1 (espacio grueso de conteo; caveat de accion-mas-rica).
from supply_chain.program_o_full_des_transducer import extract_full_des_skeleton, simulate_full_des_frontier
from supply_chain.program_o_state_rich import StateRichConfiguration, state_rich_calendar
_rng_front = np.random.default_rng(12345)
def tape_baselines(seed):
    sk,_=extract_full_des_skeleton(seed=int(seed),scheduler=SCHED_PATTERN,
        regime_persistence=CELL.regime_persistence,dominant_share=CELL.dominant_share,
        downstream_freight_physics_mode="fixed_clock_physical_v1")
    cals=_rng_front.integers(0,8,size=(SAMPLED_FRONTIER,8)).astype(np.uint8)  # patrones por-lote
    fr=simulate_full_des_frontier(skeleton=sk,scheduler=SCHED_PATTERN,calendars=cals)
    best_sampled=float(fr["ret_visible"].max())
    cal,_=state_rich_calendar(skeleton=sk.as_dict(),scheduler=SCHED_COUNT,
        config=StateRichConfiguration("belief_mpc",3),regime_persistence=0.75,dominant_share=0.90)
    cm=simulate_full_des_frontier(skeleton=sk,scheduler=SCHED_COUNT,calendars=np.asarray([cal],np.uint8))
    return dict(best_sampled=best_sampled,classical=float(cm["ret_visible"][0]),
                classical_worst=float(cm["worst_product_fill"][0]))
EVAL_SEEDS=list(range(EVAL_TAPE_START,EVAL_TAPE_END+1))
print(f"baselines en {len(EVAL_SEEDS)} tapes ({SAMPLED_FRONTIER} patrones muestreados c/u)...")
BASE={}
for i,s in enumerate(EVAL_SEEDS):
    BASE[s]=tape_baselines(s)
    if (i+1)%max(1,len(EVAL_SEEDS)//4)==0: print(f"  {i+1}/{len(EVAL_SEEDS)}")
print("clasico medio:",round(np.mean([BASE[s]['classical'] for s in EVAL_SEEDS]),5),
      "| frontera-muestreada media:",round(np.mean([BASE[s]['best_sampled'] for s in EVAL_SEEDS]),5))

## 8) Evalúa y veredicto  ·  ⛔ **NO CAMBIAR la lógica**

In [ ]:

def run_policy(model, seed):
    def _mk():
        e=PerBatchRetEnv(seed,seed+2,cell=CELL)
        return Monitor(BoxToDiscrete(e,2) if ALGORITHM in CONTINUOUS else e)
    venv=VecFrameStack(DummyVecEnv([_mk]),n_stack=FRAME_STACK)
    obs=venv.reset(); state=None; starts=np.ones((1,),bool); info=[{}]; done=[False]
    while not done[0]:
        if IS_RECURRENT: a,state=model.predict(obs,state=state,episode_start=starts,deterministic=True)
        else: a,_=model.predict(obs,deterministic=True)
        obs,r,done,info=venv.step(a); starts=done
    m=info[0]["metrics"]
    return dict(ret=float(m["ret_visible"]),worst=float(m["worst_product_fill"]),lost=float(m["lost_orders"]))
rows=[]
for s in EVAL_SEEDS:
    r=run_policy(model,s); b=BASE[s]
    rows.append(dict(seed=s,model_ret=r["ret"],best_sampled=b["best_sampled"],classical=b["classical"],
        H_learned_sampled=r["ret"]-b["best_sampled"], H_neural=r["ret"]-b["classical"],
        worst_delta=r["worst"]-b["classical_worst"]))
df=pd.DataFrame(rows)
def lcb(x,n=10000,s=7):
    x=np.asarray(x); rng=np.random.default_rng(s); return float(np.quantile(rng.choice(x,(n,len(x)),replace=True).mean(1),0.025))
Hn=df.H_neural.mean(); Hn_lcb=lcb(df.H_neural.values); w_lcb=lcb(df.worst_delta.values); fav=float((df.H_neural>0).mean()*100)
rp=FROZEN_RECURRENTPPO[CELL_ID]
print("="*72); print(f"  VARIANTE POR-LOTE:  {ALGORITHM} + {ARCH_VARIANT}  |  {CELL_ID}  |  {len(df)} tapes  |  2^24 acciones")
print("="*72)
print(f"  H_learned (vs frontera MUESTREADA {SAMPLED_FRONTIER}):  {df.H_learned_sampled.mean():+.5f}   (cota, no exacta)")
print(f"  H_neural  (vs clasico belief-MPC)             :  {Hn:+.5f}  (LCB95 {Hn_lcb:+.5f})")
print(f"     [en el entorno GRUESO, RecurrentPPO saco H_neural = {rp:+.5f} (empate)]")
print(f"  favorables vs clasico:  {fav:.1f}%   |   worst-product delta LCB95: {w_lcb:+.5f} (barra >= {WORST_PRODUCT_MARGIN})")
print("-"*72)
beat_classical = Hn>0; beat_recurrent = Hn>rp; pass_bar=(Hn_lcb>=WIN_BAR_H_NEURAL_LCB) and (w_lcb>=WORST_PRODUCT_MARGIN)
print(f"  [{'SI' if beat_classical else 'NO'}] le ganas al clasico en media (H_neural>0)  <- la señal de la hipotesis de David")
print(f"  [{'SI' if beat_recurrent else 'NO'}] superas el empate de RecurrentPPO (H_neural mayor que {rp:+.5f})")
print(f"  [{'SI' if pass_bar else 'NO'}] pasas la BARRA (LCB95(H_neural)>=+0.01 y equidad OK)")
print("="*72)
if pass_bar:
    print("  *** SEÑAL FUERTE: las decisiones finas SI abren margen. Siguiente: MPC por-lote justo + frontera")
    print("      muestreada rigurosa + seeds selladas para confirmarlo. (Caveat: el clasico aun es del espacio grueso.)")
elif beat_classical:
    print("  Le ganas al clasico en media (con acciones mas ricas), pero aun no con LCB95>=+0.01. Sube seeds/timesteps.")
else:
    print("  Todavia no. Ni con acciones por-lote RL supera al clasico grueso -> refuerza que el problema, no la")
    print("  granularidad, es lo que limita (consistente con el resultado principal RL~MPC).")
display(df.describe()[["model_ret","H_learned_sampled","H_neural","worst_delta"]])

## 9) Guardar

In [ ]:

out=Path("scresia_perbatch_outputs"); out.mkdir(exist_ok=True)
tag=f"{ALGORITHM.lower()}_{ARCH_VARIANT}_{CELL_ID}_{RUN_PROFILE}"
df.to_csv(out/f"eval_{tag}.csv",index=False)
(out/f"summary_{tag}.json").write_text(json.dumps(dict(variant="per_batch_24_decisions_2^24",
    algorithm=ALGORITHM,arch=ARCH_VARIANT,cell=CELL_ID,n_tapes=len(df),
    H_learned_sampled=df.H_learned_sampled.mean(),H_neural=Hn,H_neural_lcb95=Hn_lcb,
    worst_delta_lcb95=w_lcb,beat_classical=bool(beat_classical),pass_win_bar=bool(pass_bar),
    caveat="frontera muestreada (no exacta); clasico en espacio grueso (accion-mas-rica para RL)"),indent=2))
try: model.save(out/f"model_{tag}")
except Exception as e: print("save:",e)
print("guardado en",out.resolve())

---
**Recordatorio honesto.** Este notebook prueba la hipótesis de David (decisiones finas → margen para
RL). Si RL le gana al clásico aquí, es una **señal**, no una victoria sellada: el clásico todavía juega
en el espacio grueso, y la frontera es muestreada. Un claim fuerte necesitaría un **MPC por-lote justo**
(mismas armas) y frontera muestreada rigurosa. Pero si RL NO le gana ni con acciones ricas, refuerza el
resultado principal (RL≈MPC no es por falta de granularidad).